# 🚀 Google Colab Live AI Video API Server (Authenticated Ngrok)
Bu notebook, ngrok hesabınızla kimlik doğrulaması yapar ve kesintisiz canlı API adresi üretir.

In [ ]:
# 1. Gerekli Kütüphanelerin Kurulumu ve Ngrok Authtoken Kurulumu
!pip install -q diffusers transformers accelerate torch torchvision imageio-ffmpeg fastapi uvicorn pyngrok nest_asyncio hf_transfer
import os, torch
from pyngrok import ngrok

token = '38xGdAkRAFGU3jq422KtIiDsYwT_3qPnZbUX3hH15Y782Ko4N'
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN') or token
except Exception:
    pass

ngrok.set_auth_token(token)
print('✅ NGROK Authtoken Başarıyla Doğrulandı!')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Kurulum Tamamlandı! Kullanılan Donanım: {device.upper()}')

In [ ]:
# 2. Text-to-Video AI Modelinin Yüklenmesi
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

print(f'🚀 AI Video Modeli {device.upper()} Üzerinde Yükleniyor...')
dtype = torch.float16 if device == 'cuda' else torch.float32
pipe = DiffusionPipeline.from_pretrained(
    'damo-vilab/text-to-video-ms-1.7b',
    torch_dtype=dtype
)
pipe = pipe.to(device)
if device == 'cuda':
    pipe.enable_attention_slicing()
print('✅ AI Video Modeli Başarıyla Yüklendi!')

In [ ]:
# 3. Canlı ngrok API Web Sunucusu
!fuser -k 8000/tcp || true
import time, threading, uuid
from fastapi import FastAPI, Response
from pydantic import BaseModel
import uvicorn
import nest_asyncio

app = FastAPI()
jobs = {}

class VideoRequest(BaseModel):
    prompt: str
    niche: str = 'minecraft'
    width: int = 256
    height: int = 448

@app.get('/')
def health_check():
    return {'status': 'online', 'model': 'ModelScope 1.7B', 'device': device}

def process_video_job(job_id: str, req: VideoRequest):
    try:
        jobs[job_id] = {'status': 'processing'}
        print(f'🎬 [{job_id}] Video üretimi başladı: {req.prompt}')
        video_frames = pipe(
            prompt=req.prompt,
            num_inference_steps=20,
            height=448,
            width=256,
            num_frames=16
        ).frames[0]
        
        out_path = f'/content/video_{job_id}.mp4'
        export_to_video(video_frames, out_path, fps=16)
        jobs[job_id] = {'status': 'completed', 'file_path': out_path}
        print(f'✅ [{job_id}] Video üretimi tamamlandı!')
    except Exception as e:
        jobs[job_id] = {'status': 'failed', 'error': str(e)}
        print(f'❌ [{job_id}] Hata: {e}')

@app.post('/start_generation')
def start_generation(req: VideoRequest):
    job_id = str(uuid.uuid4())[:8]
    threading.Thread(target=process_video_job, args=(job_id, req), daemon=True).start()
    return {'status': 'started', 'job_id': job_id}

@app.get('/job_status/{job_id}')
def job_status(job_id: str):
    return jobs.get(job_id, {'status': 'not_found'})

@app.get('/download_video/{job_id}')
def download_video(job_id: str):
    job_info = jobs.get(job_id, {})
    if job_info.get('status') == 'completed':
        with open(job_info['file_path'], 'rb') as f:
            return Response(content=f.read(), media_type='video/mp4')
    return Response(status_code=404, content='Not ready')

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

nest_asyncio.apply()
public_url = ngrok.connect(8000)
url_str = str(public_url)
print('====================================================')
print('🚀 CANLI GOOGLE COLAB API URL ADRESİNİZ:')
print(url_str)
print(f'COLAB_API_URL={url_str}')
print('====================================================')

# ✅ URL'yi otomatik Telegram'a gönder
import requests as _req
BOT_TOKEN = '8709377467:AAFMfhaHuGND6rgM4Kxr5Brklvqwn56Ezko'
CHAT_ID = '1215543640'
msg = f'✅ Colab Sunucusu Başladı!\n\nURL: {url_str}\n\nBu URL otomatik kullanılacak.'
_req.post(f'https://api.telegram.org/bot{BOT_TOKEN}/sendMessage', data={'chat_id': CHAT_ID, 'text': msg})
print('✅ URL Telegram\'a gönderildi!')